# Methoden en Technieken 2025-2026 -- Blok 3

## Datapunt Opdracht 3a

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld** **machine learning**, deep learning, kennisrepresentatie, computer vision en **natural language processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability, explainability, **model complexity** en resource demand.


## De opdracht

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

In [ ]:
# %pip install pandas surprise numpy==1.26.4 scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [35]:
import pandas as pd
import numpy as np
import math
import warnings

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV as SurpriseGridSearchCV
from surprise.model_selection import KFold as SurpriseKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# Data prep

In [36]:
ratings = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
    delimiter='::', engine='python', header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)
print(f"Loaded {len(ratings):,} ratings")

Loaded 921,398 ratings


## Add movie data

TODO rational

In [37]:
items_raw = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat',
    delimiter='::', engine='python', header=None,
    names=['movie_id', 'title_raw', 'genres_raw'],
    encoding='utf-8'
)

# Extract year from title, e.g. "Toy Story (1995)" → 1995
items_raw['year'] = items_raw['title_raw'].str.extract(r'\((\d{4})\)').astype(float)
items_raw['title'] = items_raw['title_raw'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True)

# Split genre string into a list (literal '|') and guard against missing values
items_raw['genres_raw'] = items_raw['genres_raw'].fillna('')
items_raw['genres'] = items_raw['genres_raw'].str.split('|', regex=False)
items_raw['genres'] = items_raw['genres'].apply(lambda g: [x for x in g if x])

# Ensure unique movie metadata per movie_id
items = (
    items_raw[['movie_id', 'title', 'year', 'genres']]
    .drop_duplicates(subset=['movie_id'], keep='last')
    .reset_index(drop=True)
)
print(f"Loaded {len(items):,} unique movies")

Loaded 38,013 unique movies


In [38]:
items

,movie_id,title,year,genres
0,8,Edison Kinetoscopic Record of a Sneeze,1894.0,"[Documentary, Short]"
1,10,La sortie des usines Lumière,1895.0,"[Documentary, Short]"
2,12,The Arrival of a Train,1896.0,"[Documentary, Short]"
3,25,The Oxford and Cambridge University Boat Race,1895.0,[]
4,91,Le manoir du diable,1896.0,"[Short, Horror]"
...,...,...,...,...
38008,15711402,Les rois de l&x27;arnaque,2021.0,"[Crime, Documentary]"
38009,15831978,Cash,2021.0,[]
38010,15839820,Sompoy,2021.0,"[Comedy, Romance]"
38011,15842076,The Making of &x27;Rocky vs. Drago&x27;,2021.0,[Documentary]


De bedoeling is om een aanbevelings-systeem te bouwen dat voor elke willekeurige gebruiker in het systeem drie films aanbeveelt. Probeer de aanbeveling zo persoonlijk mogelijk te maken.
* Kies een model en verantwoord deze keuze.
* Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze.
* Evalueer het model.
* Geef concrete suggesties om het model te verbeteren. Je hoeft deze verbeteringen niet uit te voeren.
* Bespreek voor- en nadelen van het model dat je hebt gemaakt.

In [39]:
ratings

,user_id,movie_id,rating,timestamp
0,1,114508,8,1381006850
1,2,499549,9,1376753198
2,2,1305591,8,1376742507
3,2,1428538,1,1371307089
4,3,75314,1,1595468524
...,...,...,...,...
921393,71705,9893250,10,1613857551
921394,71705,9898858,3,1585958452
921395,71706,172495,10,1587107015
921396,71706,414387,10,1587107852


In [40]:
# ── Merge ratings with movie metadata ─────────────────────────────────────────
df = ratings.merge(items, on='movie_id', how='inner')

# Ensure rating is numeric and within the 0–10 scale
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[(df['rating'] >= 0) & (df['rating'] <= 10)].dropna(subset=['rating', 'genres'])

print(f"Merged dataset: {len(df):,} ratings  |  rating range {df['rating'].min():.0f}–{df['rating'].max():.0f}")
df.head()

Merged dataset: 921,398 ratings  |  rating range 0–10


,user_id,movie_id,rating,timestamp,title,year,genres
0,1,114508,8,1381006850,Species,1995.0,"[Action, Horror, Sci-Fi, Thriller]"
1,2,499549,9,1376753198,Avatar,2009.0,"[Action, Adventure, Fantasy, Sci-Fi]"
2,2,1305591,8,1376742507,Mars Needs Moms,2011.0,"[Animation, Adventure, Family, Sci-Fi]"
3,2,1428538,1,1371307089,Hansel &amp; Gretel: Witch Hunters,2013.0,"[Action, Fantasy, Horror]"
4,3,75314,1,1595468524,Taxi Driver,1976.0,"[Crime, Drama]"


## 2. Data Preparation — Decisions & Rationale

#### Data preparation
Gebruikers met zeer weinig interacties worden gefilterd voordat het recommender model wordt getraind, omdat deze gebruikers onvoldoende informatie bieden om persoonlijke aanbevelingen te maken [1, p.229] (H9: “You’ll usually filter users away with only a small number of interactions in your data”).

Daarnaast wordt de dataset eerst opgeschoond voordat het model wordt geëvalueerd, omdat gebruikers met weinig ratings weinig bijdragen aan het leerproces van het model [1, p.236] (H9: “Most data sets need a bit of housekeeping before you evaluate them… all the users who rated only a few items don’t help the recommender much”).

#### Minimum aantal ratings voor collaborative filtering

Collaborative filtering algoritmen vereisen voldoende overlap tussen gebruikers om gelijkenissen te berekenen; gebruikers met slechts één rating leveren daarom geen bruikbare informatie voor dit type algoritme [1, p.229] (H9: “The algorithm won’t work for users with only one rating. A user with only one registered rating also won’t do anything for collaborative filtering algorithms”).

#### Train/test datasplitsing

Voor de evaluatie van recommender systemen wordt de dataset gesplitst op basis van tijd zodat het model alleen leert van eerdere interacties en voorspellingen maakt voor latere interacties [1, p.233] (H9: “From a recommender system evaluation’s point of view, it makes more sense to split the data based on time”).

Daarnaast kan een per-gebruiker splitsing worden gebruikt waarbij de eerste ratings van een gebruiker in de trainingsset gaan en de rest in de testset, zodat elke gebruiker in de trainingsdata voorkomt [1, p.233–234] (H9: “The ratings will be divided by taking the first n ratings in the training set and the rest in the testing set”).

#### Baseline recommender

Bij het evalueren van een recommender systeem is het belangrijk om de prestaties te vergelijken met een eenvoudige baseline, zoals een popularity-based recommender [1, p.239] (H9: “Before you evaluate your new recommender, you should evaluate it on a simple recommender that, for example, always recommends the most popular items”).

#### Evaluatiemetriek

De kwaliteit van het model kan worden gemeten met RMSE, een foutmaat die grote voorspellingsfouten zwaarder bestraft dan kleinere fouten [1, p.224] (H9: “The RMSE will put big penalties on big errors”).

#### Interpretatie van Precision-scores

Bij recommender systemen is het normaal dat de eerste precisie-scores relatief laag zijn, omdat het voorspellen van relevante aanbevelingen moeilijk is [1, p.239] (H9: “Everybody has received disturbingly low numbers the first time they ran an evaluation on their recommender. (I got a precision that was something like 0.063 the first time around.)”).

#### Cross-validation for evaluation

Om de prestaties van een recommender systeem betrouwbaarder te evalueren kan k-fold cross-validation worden gebruikt, waarbij het model meerdere keren wordt getraind en geëvalueerd op verschillende subsets van de data [1, p.236] (H9: “The implementation uses a k-fold cross-validation.”).

#### Parameter tuning

Bij het optimaliseren van een recommender model moeten verschillende parameters systematisch worden getest om te bepalen welke instellingen de beste evaluatiescores opleveren [1, p.242] (H9: “You want these parameters to be set so they return the best evaluation. To do that, pick one and then run the evaluator for a range of values.”).

#### Minimum overlap voor similarity

Voor collaborative filtering is voldoende overlap tussen ratings van verschillende gebruikers noodzakelijk om gelijkenissen tussen items of gebruikers betrouwbaar te berekenen [1, p.242] (H9: “Looks only at similarities where more than min_overlap users have rated both movies”).

| Step | Decision | Rationale |
|---|---|---|
| **Filter users with < 5 ratings** | Remove users with fewer than 5 ratings | These users provide almost no signal for collaborative filtering and create unstable training examples. The recommender cannot learn meaningful preferences from them. With the given-n split (first 5 → train), every remaining user is guaranteed to have training *and* test data. |
| **Filter movies with < 3 ratings** | Remove movies with fewer than 3 ratings | Extremely rare movies create isolated columns in the user–item matrix, increasing sparsity and potentially destabilizing matrix factorization. Filtering them slightly improves factorization quality. |
| **Chronological ordering** | Sort each user's ratings by timestamp *before* splitting | Random splitting would allow the model to train on future ratings while predicting past ones (temporal leakage). Chronological ordering ensures the model predicts future behaviour based on past behaviour only. |
| **Given-n split protocol** | First 5 ratings per user go to train; the rest go to test | This guarantees every evaluated user has a fixed minimum of training data *and* at least one test rating. It avoids the problem of users appearing only in the test set and provides a realistic recommendation scenario. |
| **Remove test-only users** | After splitting, remove any user that appears only in the test set | A recommender cannot generate predictions for users it has never seen during training. Keeping them would distort evaluation metrics. |
| **Identify cold users (< 20 training ratings)** | Flag cold users explicitly after filtering | Collaborative filtering relies on overlapping preferences. Users with very few interactions provide weak signals. Cold users are handled separately at recommendation time (popularity fallback). |
| **Normalize ratings** | Subtract each user's mean rating before collaborative filtering training | Some users consistently rate higher or lower than others. Centering removes this personal bias and improves similarity calculations and factorization quality. |

In [41]:
# ── Exploratory Checks ────────────────────────────────────────────────────────
n_users  = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()
n_ratings = len(df)
avg_per_user = n_ratings / n_users

print(f"Users:                {n_users:,}")
print(f"Movies:               {n_movies:,}")
print(f"Total ratings:        {n_ratings:,}")
print(f"Avg ratings / user:   {avg_per_user:.2f}")

# ── Step 1: Remove users with fewer than 5 ratings ───────────────────────────
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= 5].index
df = df[df['user_id'].isin(valid_users)].copy()

print(f"\nAfter removing users with < 5 ratings:")
print(f"  Users:   {df['user_id'].nunique():,}")
print(f"  Ratings: {len(df):,}")

# ── Step 2: Remove movies with fewer than 3 ratings ──────────────────────────
movie_counts = df.groupby('movie_id').size()
valid_movies = movie_counts[movie_counts >= 3].index
df = df[df['movie_id'].isin(valid_movies)].copy()

print(f"\nAfter removing movies with < 3 ratings:")
print(f"  Users:   {df['user_id'].nunique():,}")
print(f"  Movies:  {df['movie_id'].nunique():,}")
print(f"  Ratings: {len(df):,}")

Users:                71,707
Movies:               38,013
Total ratings:        921,398
Avg ratings / user:   12.85

After removing users with < 5 ratings:
  Users:   23,805
  Ratings: 845,154

After removing movies with < 3 ratings:
  Users:   23,802
  Movies:  16,384
  Ratings: 819,491


## 3. Data split — Given-n Protocol (chronological)

For every user, ratings are first **sorted by timestamp**. The first **5** ratings become the training set (**given-n**, $n = 5$); every subsequent rating is placed in the test set. After splitting, any user that ended up only in the test set is removed to guarantee valid evaluation. Cold users (fewer than 20 training ratings) are flagged for special handling at recommendation time.

In [42]:
# ── Given-n train / test split (chronological) ────────────────────────────────
GIVEN_N = 5          # number of ratings per user that go to training

def given_n_split(data, n=GIVEN_N):
    """
    Per-user chronological split: the first *n* ratings (by timestamp)
    go to train, the remaining ratings go to test.
    Users with ≤ n ratings go entirely to train (no test data for them).
    """
    train_parts, test_parts = [], []
    for _, group in data.groupby('user_id'):
        g = group.sort_values('timestamp')
        train_parts.append(g.iloc[:n])
        if len(g) > n:
            test_parts.append(g.iloc[n:])
    train = pd.concat(train_parts).reset_index(drop=True)
    test  = pd.concat(test_parts).reset_index(drop=True) if test_parts else pd.DataFrame(columns=data.columns)
    return train, test

train_df, test_df = given_n_split(df, n=GIVEN_N)

# ── Ensure every test user also appears in training ──────────────────────────
train_user_set = set(train_df['user_id'].unique())
test_df = test_df[test_df['user_id'].isin(train_user_set)].copy()

print(f"Train: {len(train_df):,} ratings  ({train_df['user_id'].nunique():,} users)")
print(f"Test:  {len(test_df):,} ratings  ({test_df['user_id'].nunique():,} users)")
print(f"All test users appear in training: {set(test_df['user_id'].unique()).issubset(train_user_set)}")

# ── Identify cold users (< 20 training ratings) ─────────────────────────────
COLD_THRESHOLD = 20
user_train_counts = train_df.groupby('user_id').size().to_dict()
cold_users  = {u for u, c in user_train_counts.items() if c < COLD_THRESHOLD}
warm_users  = {u for u, c in user_train_counts.items() if c >= COLD_THRESHOLD}

print(f"\nCold users (< {COLD_THRESHOLD} training ratings): {len(cold_users):,}")
print(f"Warm users (≥ {COLD_THRESHOLD} training ratings): {len(warm_users):,}")

Train: 118,717 ratings  (23,802 users)
Test:  700,774 ratings  (21,205 users)
All test users appear in training: True

Cold users (< 20 training ratings): 23,802
Warm users (≥ 20 training ratings): 0


## Data exploration

TODO

## Base model Popularity Recommender

For every movie, compute:
$$\text{popularity\_score} = \overline{r} \cdot \ln(1 + n)$$
where $\overline{r}$ is the average rating and $n$ the number of ratings. This baseline is also used for **cold-start users**.

In [43]:
# ── Baseline: Popularity Recommender ──────────────────────────────────────────
movie_stats = train_df.groupby('movie_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# popularity_score = avg_rating × log(1 + num_ratings)
movie_stats['popularity_score'] = (
    movie_stats['avg_rating'] * np.log1p(movie_stats['num_ratings'])
)
movie_stats = movie_stats.merge(
    items[['movie_id', 'title', 'genres']], on='movie_id', how='left'
).sort_values('popularity_score', ascending=False).reset_index(drop=True)

# Pre-compute fast-lookup dicts
popularity_dict = movie_stats.set_index('movie_id')['popularity_score'].to_dict()
movie_avg_dict  = movie_stats.set_index('movie_id')['avg_rating'].to_dict()

print("Top-10 most popular movies (baseline):\n")
print(
    movie_stats[['title', 'avg_rating', 'num_ratings', 'popularity_score']]
    .head(10)
    .to_string(index=False)
)

Top-10 most popular movies (baseline):

                   title  avg_rating  num_ratings  popularity_score
                    1917    8.573209          963         58.907300
            Interstellar    9.133721          516         57.067880
The Shawshank Redemption    9.561688          308         54.820422
        Django Unchained    8.645390          564         54.784330
                 Gravity    8.254808          624         53.142402
                   Joker    9.104167          336         52.987005
            Gisaengchung    8.424242          528         52.828327
 The Wolf of Wall Street    8.527352          457         52.245972
           Hacksaw Ridge    8.978593          327         52.013113
 Star Trek Into Darkness    8.235185          540         51.827473


## Model Architecture — Decisions & Rationale

The recommender is a **hybrid model** that combines collaborative filtering (SVD) with content-based filtering (genre cosine similarity). Below is a summary of every design choice and its motivation.

| Component | Choice | Rationale |
|---|---|---|
| **Content-based filtering** | One-hot genre vectors + cosine similarity between user profile and movie | Genre information is always available, making content-based scoring robust against cold-start items. User profiles are built by averaging vectors of highly-rated movies (rating ≥ 7). |
| **Collaborative filtering** | SVD matrix factorization (Surprise) with user-mean-centered ratings | SVD captures latent factors from user–item interactions and generalises well on sparse data. Centering ratings by user mean removes personal rating bias, improving factorization quality. |
| **Hyperparameter tuning** | Grid search over `n_factors`, `n_epochs`, `lr_all`, `reg_all` | Matrix factorization performance is sensitive to these parameters. Systematic tuning avoids arbitrary defaults and improves RMSE. |
| **Hybrid scoring** | `final_score = 0.8 × collab + 0.2 × content` | Collaborative filtering typically provides stronger personalisation signals; content similarity acts as a regulariser and helps with cold items. |
| **Cold-start handling** | Users with < 20 training ratings → popularity fallback; movies with < 5 ratings → global average collab score | Cold users and items lack sufficient interaction data for reliable collaborative predictions. Popularity is a reasonable non-personalised fallback. |
| **Diversity / serendipity** | Top 2 by hybrid score; 3rd pick from a different genre cluster | Prevents repetitive genre-locked recommendations and exposes users to broader content. |
| **Popularity baseline** | `popularity_score = avg_rating × log(1 + num_ratings)` | Combines quality (avg rating) and quantity (num ratings) so movies with few but extreme ratings do not dominate. Serves as the benchmark to beat. |
| **Evaluation** | RMSE (prediction accuracy) + Precision@3 (ranking quality, relevance threshold ≥ 7) on 500+ users; optional 5-fold cross-validation | RMSE measures rating prediction error; Precision@3 measures top-list quality. A larger evaluation sample reduces variance. k-fold CV provides a more reliable performance estimate. |

## 5. Content-Based Filtering (genre similarity)

1. One-hot encode the genre lists into a binary feature matrix.
2. Build a **user genre profile** by averaging the genre vectors of movies rated ≥ 7.
3. Compute **cosine similarity** between each user profile and every movie's genre vector.

In [44]:
# ── Content-Based: genre one-hot matrix + user profiles ───────────────────────

# Ensure genres are always list-like
items['genres'] = items['genres'].apply(
    lambda g: g if isinstance(g, list) else ([] if pd.isna(g) else [str(g)])
)

# One-hot encode genres into a binary movie×genre matrix
mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    mlb.fit_transform(items['genres']),
    columns=mlb.classes_,
    index=items['movie_id']
)
# Safety: enforce unique index
genre_matrix = genre_matrix.groupby(level=0).max()

# Drop placeholder genre if it exists
if '(no genres listed)' in genre_matrix.columns:
    genre_matrix.drop(columns=['(no genres listed)'], inplace=True)

all_genre_names = list(genre_matrix.columns)
print(f"Genre features ({len(all_genre_names)}): {', '.join(all_genre_names)}")

# ── Build user genre profiles (avg genre vector of movies rated ≥ 7) ─────────
def build_user_profiles(train_data, genre_mat, threshold=7):
    profiles = {}
    for uid, grp in train_data.groupby('user_id'):
        liked = grp[grp['rating'] >= threshold]['movie_id']
        vecs  = genre_mat.loc[genre_mat.index.intersection(liked)]
        if len(vecs) == 0:                       # fallback: use all rated movies
            vecs = genre_mat.loc[genre_mat.index.intersection(grp['movie_id'])]
        profiles[uid] = vecs.mean().values if len(vecs) > 0 else np.zeros(genre_mat.shape[1])
    return profiles

user_profiles = build_user_profiles(train_df, genre_matrix)

# ── Pre-compute full cosine-similarity matrix (users × movies) ───────────────
user_ids_ordered  = list(user_profiles.keys())
user_profile_mat  = np.array([user_profiles[u] for u in user_ids_ordered])
movie_ids_ordered = genre_matrix.index.tolist()
genre_mat_values  = genre_matrix.values

cs_matrix = cosine_similarity(user_profile_mat, genre_mat_values)   # shape (n_users, n_movies)

user_idx_map  = {u: i for i, u in enumerate(user_ids_ordered)}
movie_idx_map = {m: j for j, m in enumerate(movie_ids_ordered)}

def content_score_lookup(user_id, movie_id):
    """O(1) cosine similarity between a user's genre profile and a movie."""
    ui = user_idx_map.get(user_id)
    mi = movie_idx_map.get(movie_id)
    if ui is None or mi is None:
        return 0.0
    return float(cs_matrix[ui, mi])

print(f"User profiles built for {len(user_profiles):,} users")
print(f"Content-similarity matrix shape: {cs_matrix.shape}")

Genre features (28): Action, Adult, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, Film-Noir, Game-Show, History, Horror, Music, Musical, Mystery, News, Reality-TV, Romance, Sci-Fi, Short, Sport, Talk-Show, Thriller, War, Western
User profiles built for 23,802 users
Content-similarity matrix shape: (23802, 38013)


## 6. Collaborative Filtering — SVD with Rating Centering & Hyperparameter Tuning

**Rating centering:** Before training, each rating is centered by subtracting the user's mean rating. This removes personal scale bias (some users rate consistently high/low). The user means are stored so predictions can be un-centered later.

**Hyperparameter tuning:** A grid search over `n_factors` ∈ {50, 100, 150}, `n_epochs` ∈ {20, 30}, `lr_all` ∈ {0.002, 0.005, 0.01}, and `reg_all` ∈ {0.02, 0.05, 0.1} is performed using Surprise's built-in `GridSearchCV` (3-fold). The best parameters are then used to train the final model on the full training set.

In [45]:
# ── Rating centering ──────────────────────────────────────────────────────────
user_mean_rating = train_df.groupby('user_id')['rating'].mean().to_dict()
train_centered = train_df.copy()
train_centered['rating_centered'] = train_centered.apply(
    lambda r: r['rating'] - user_mean_rating.get(r['user_id'], 0), axis=1
)

# The centered scale can be negative; compute the actual range for Surprise
centered_min = train_centered['rating_centered'].min()
centered_max = train_centered['rating_centered'].max()
print(f"Centered rating range: [{centered_min:.2f}, {centered_max:.2f}]")

# ── Surprise dataset from centered ratings ────────────────────────────────────
reader_centered = Reader(rating_scale=(centered_min, centered_max))
surprise_data_centered = Dataset.load_from_df(
    train_centered[['user_id', 'movie_id', 'rating_centered']], reader_centered
)

# ── Grid search for best SVD hyperparameters (3-fold CV) ─────────────────────
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs':  [20, 30],
    'lr_all':    [0.002, 0.005, 0.01],
    'reg_all':   [0.02, 0.05, 0.1],
}

print("Running SVD grid search (3-fold CV) — this may take a few minutes …")
gs = SurpriseGridSearchCV(SVD, param_grid, measures=['rmse'], cv=3,
                          refit=False, n_jobs=-1)
gs.fit(surprise_data_centered)

best_params = gs.best_params['rmse']
print(f"\nBest RMSE (CV): {gs.best_score['rmse']:.4f}")
print(f"Best params:    {best_params}")

# ── Train the final SVD model with best parameters on the full training set ──
trainset_centered = surprise_data_centered.build_full_trainset()

svd = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)
svd.fit(trainset_centered)

print(f"\nFinal SVD model trained — factors={svd.n_factors}, epochs={svd.n_epochs}")
print(f"Internal trainset: {trainset_centered.n_users} users, "
      f"{trainset_centered.n_items} items, {trainset_centered.n_ratings} ratings")

Centered rating range: [-8.00, 7.00]
Running SVD grid search (3-fold CV) — this may take a few minutes …

Best RMSE (CV): 1.4233
Best params:    {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.002, 'reg_all': 0.05}

Final SVD model trained — factors=50, epochs=20
Internal trainset: 23802 users, 10239 items, 118717 ratings


## 7. Hybrid Scoring Function & Cold-Start Handling

$$\text{final\_score} = 0.8 \times \text{collaborative\_score} + 0.2 \times \text{content\_score}$$

Because the SVD was trained on **user-mean-centered** ratings, every collaborative prediction is un-centered by adding the user's mean rating back.

**Cold-start rules:**
- **Users** with < 20 training ratings → skip collaborative filtering, use the popularity model with genre diversity.
- **Movies** with < 5 training ratings → substitute a neutral collaborative score (user's mean rating) so the content signal dominates.

In [46]:
# ── Hybrid scoring with cold-start handling ───────────────────────────────────
COLLAB_WEIGHT  = 0.8
CONTENT_WEIGHT = 0.2

# Pre-compute useful lookups
movie_train_counts = train_df.groupby('movie_id').size().to_dict()
user_rated_sets    = train_df.groupby('user_id')['movie_id'].apply(set).to_dict()
all_movie_ids      = set(items['movie_id'].unique())
global_avg_rating  = train_df['rating'].mean()

def _batch_svd_predict(user_id, movie_ids):
    """Vectorised SVD predictions (un-centered) using internal factor matrices."""
    n = len(movie_ids)
    u_mean = user_mean_rating.get(user_id, global_avg_rating)
    try:
        iuid = svd.trainset.to_inner_uid(user_id)
    except ValueError:
        return np.full(n, u_mean)

    pu = svd.pu[iuid]
    bu = svd.bu[iuid]
    mu = svd.trainset.global_mean

    preds  = np.full(n, u_mean)          # fallback for unknown movies
    valid  = np.zeros(n, dtype=bool)
    iiids  = np.zeros(n, dtype=int)
    for i, mid in enumerate(movie_ids):
        try:
            iiids[i] = svd.trainset.to_inner_iid(mid)
            valid[i] = True
        except ValueError:
            pass

    if valid.any():
        qi_batch = svd.qi[iiids[valid]]
        bi_batch = svd.bi[iiids[valid]]
        # SVD predicts centered ratings → un-center by adding user mean
        preds[valid] = (mu + bu + bi_batch + qi_batch @ pu) + u_mean

    return np.clip(preds, 0, 10)

def get_hybrid_score(user_id, movie_id):
    """Single-pair hybrid score (used during evaluation)."""
    is_cold_user = user_id in cold_users
    if is_cold_user:
        return popularity_dict.get(movie_id, 0.0)

    u_mean = user_mean_rating.get(user_id, global_avg_rating)
    is_cold_movie = movie_train_counts.get(movie_id, 0) < 5
    if is_cold_movie:
        collab = u_mean                              # neutral fallback
    else:
        try:
            iuid = svd.trainset.to_inner_uid(user_id)
            iiid = svd.trainset.to_inner_iid(movie_id)
            collab_centered = (svd.trainset.global_mean
                               + svd.bu[iuid] + svd.bi[iiid]
                               + np.dot(svd.pu[iuid], svd.qi[iiid]))
            collab = np.clip(collab_centered + u_mean, 0, 10)
        except ValueError:
            collab = u_mean

    cb = content_score_lookup(user_id, movie_id) * 10.0
    return COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

print(f"Hybrid weights: collab={COLLAB_WEIGHT}, content={CONTENT_WEIGHT}")
print(f"Cold-start threshold: user < {COLD_THRESHOLD} ratings → popularity fallback")
print(f"Cold-start movie threshold: < 5 ratings → user mean ({global_avg_rating:.2f} global avg)")

Hybrid weights: collab=0.8, content=0.2
Cold-start threshold: user < 20 ratings → popularity fallback
Cold-start movie threshold: < 5 ratings → user mean (7.61 global avg)


## 8. Recommendation Function with Serendipity & Diversity

For each user:
1. Score all unrated movies with the hybrid model.
2. Pick the **top 2** by score.
3. Pick a **diverse 3rd** movie whose genres are *not* a subset of the first two — this adds serendipity.

Cold-start users (fewer than 20 training ratings) receive popularity-based picks with enforced genre diversity.

In [47]:
# ── Recommendation generation with diversity / serendipity ────────────────────

def recommend(user_id, n=3):
    """
    Return a DataFrame with the top-n recommended movies for *user_id*.
    Uses the hybrid model for warm users and the popularity model for cold-start users.
    Enforces genre diversity for the third recommendation (serendipity).
    """
    rated      = user_rated_sets.get(user_id, set())
    candidates = np.array(list(all_movie_ids - rated))
    if len(candidates) == 0:
        return pd.DataFrame(columns=['rank', 'movie_id', 'title', 'genres', 'score'])

    is_cold = user_id in cold_users

    # ── Score every candidate ────────────────────────────────────────────────
    if is_cold:
        scores = np.array([popularity_dict.get(m, 0.0) for m in candidates])
    else:
        # Collaborative scores (vectorised, un-centered)
        collab_scores = _batch_svd_predict(user_id, candidates)
        # Content scores (vectorised)
        ui = user_idx_map.get(user_id)
        if ui is not None:
            cb_scores = np.array([
                cs_matrix[ui, movie_idx_map[m]] * 10.0
                if m in movie_idx_map else 0.0
                for m in candidates
            ])
        else:
            cb_scores = np.zeros(len(candidates))
        scores = COLLAB_WEIGHT * collab_scores + CONTENT_WEIGHT * cb_scores

    # ── Rank and apply diversity rule ────────────────────────────────────────
    order = np.argsort(-scores)
    sorted_cands  = candidates[order]
    sorted_scores = scores[order]

    if len(sorted_cands) <= n or n <= 2:
        return _format_recs(list(zip(sorted_cands[:n], sorted_scores[:n])))

    if is_cold:
        return _diverse_popularity_picks(sorted_cands, sorted_scores, n)

    # Top 2 by hybrid score
    top2 = [(sorted_cands[0], sorted_scores[0]),
            (sorted_cands[1], sorted_scores[1])]

    # Genres covered by top 2
    top2_genres = set()
    for mid, _ in top2:
        if mid in movie_idx_map:
            row = genre_matrix.loc[mid]
            top2_genres.update(row[row == 1].index)

    # Diverse 3rd pick: first candidate whose genres are NOT a subset of top-2
    diverse = None
    search_limit = min(80, len(sorted_cands))
    for i in range(2, search_limit):
        mid = sorted_cands[i]
        if mid in movie_idx_map:
            mg = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if mg and not mg.issubset(top2_genres):
                diverse = (mid, sorted_scores[i])
                break
    if diverse is None:
        diverse = (sorted_cands[2], sorted_scores[2])

    return _format_recs([top2[0], top2[1], diverse])


def _diverse_popularity_picks(sorted_cands, sorted_scores, n):
    """Pick the top-n popular movies ensuring each adds at least one new genre."""
    picked = []
    seen_genres = set()
    for mid, sc in zip(sorted_cands, sorted_scores):
        if mid in movie_idx_map:
            mg = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if not mg.issubset(seen_genres) or len(picked) == 0:
                picked.append((mid, sc))
                seen_genres.update(mg)
        if len(picked) >= n:
            break
    # Fill any remaining slots from top
    if len(picked) < n:
        existing = {p[0] for p in picked}
        for mid, sc in zip(sorted_cands, sorted_scores):
            if mid not in existing:
                picked.append((mid, sc))
            if len(picked) >= n:
                break
    return _format_recs(picked[:n])


def _format_recs(picked):
    """Convert list of (movie_id, score) tuples to a presentation DataFrame."""
    rows = []
    for rank, (mid, sc) in enumerate(picked, 1):
        info = items[items['movie_id'] == mid]
        title  = info['title'].values[0] if len(info) else f"Movie {mid}"
        genres = ', '.join(info['genres'].values[0]) if len(info) and isinstance(info['genres'].values[0], list) else ''
        rows.append({
            'rank': rank, 'movie_id': int(mid),
            'title': title, 'genres': genres,
            'score': round(float(sc), 3)
        })
    return pd.DataFrame(rows)


print("recommend() function ready.")

recommend() function ready.


## 9. Evaluation

| Metric | What it measures |
|---|---|
| **RMSE** | Prediction accuracy — how close predicted ratings are to actual test ratings. |
| **Precision@3** | Ranking quality — what fraction of the top-3 recommendations are *relevant* (test rating ≥ 7)? |

Both the **hybrid** model and the **popularity baseline** are compared.

**Evaluation sample:** Precision@3 is computed on **500+** test users (instead of 300) to reduce variance in the estimate.

**5-fold cross-validation** is also performed on the centered training data to obtain a more robust RMSE estimate for the SVD component alone.

In [48]:
# ── Evaluation: RMSE, Precision@3, and k-fold CV ─────────────────────────────

# ---------- 1) RMSE — hybrid model (with un-centering) ----------
def compute_rmse_hybrid(test_data):
    uids    = test_data['user_id'].values
    mids    = test_data['movie_id'].values
    actuals = test_data['rating'].values
    preds   = np.empty(len(actuals))

    mu = svd.trainset.global_mean
    for i in range(len(actuals)):
        uid, mid = uids[i], mids[i]
        u_mean = user_mean_rating.get(uid, global_avg_rating)

        if uid in cold_users:                             # cold-start user
            preds[i] = movie_avg_dict.get(mid, global_avg_rating)
            continue
        if movie_train_counts.get(mid, 0) < 5:           # cold-start movie
            collab = u_mean
        else:
            try:
                iuid = svd.trainset.to_inner_uid(uid)
                iiid = svd.trainset.to_inner_iid(mid)
                collab_c = mu + svd.bu[iuid] + svd.bi[iiid] + np.dot(svd.pu[iuid], svd.qi[iiid])
                collab = np.clip(collab_c + u_mean, 0, 10)
            except ValueError:
                collab = u_mean
        cb = content_score_lookup(uid, mid) * 10.0
        preds[i] = COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

    preds = np.clip(preds, 0, 10)
    return math.sqrt(mean_squared_error(actuals, preds))

# ---------- 2) RMSE — popularity baseline ----------
def compute_rmse_baseline(test_data):
    actuals = test_data['rating'].values
    preds   = np.array([movie_avg_dict.get(m, global_avg_rating) for m in test_data['movie_id'].values])
    return math.sqrt(mean_squared_error(actuals, preds))

# ---------- 3) Precision@K ----------
def precision_at_k(test_data, rec_func, k=3, threshold=7):
    precisions = []
    for uid in test_data['user_id'].unique():
        relevant = set(test_data[(test_data['user_id'] == uid) & (test_data['rating'] >= threshold)]['movie_id'])
        if not relevant:
            continue
        recs = rec_func(uid, n=k)
        if recs.empty:
            precisions.append(0.0)
            continue
        hits = len(set(recs['movie_id']) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0.0

# ---------- 4) Baseline recommend function ----------
def baseline_recommend(user_id, n=3):
    rated = user_rated_sets.get(user_id, set())
    top = movie_stats[~movie_stats['movie_id'].isin(rated)].head(n)
    rows = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        g = ', '.join(r['genres']) if isinstance(r['genres'], list) else str(r['genres'])
        rows.append({'rank': rank, 'movie_id': r['movie_id'],
                     'title': r['title'], 'genres': g,
                     'score': round(r['popularity_score'], 3)})
    return pd.DataFrame(rows)

# ---------- 5) Run evaluation ----------
print("Computing RMSE on the full test set …")
rmse_hybrid   = compute_rmse_hybrid(test_df)
rmse_baseline = compute_rmse_baseline(test_df)

# Precision@3 on a larger sample of test users (500+)
sample_size    = min(500, test_df['user_id'].nunique())
sample_users   = np.random.choice(test_df['user_id'].unique(), size=sample_size, replace=False)
sample_test_df = test_df[test_df['user_id'].isin(sample_users)]

print(f"Computing Precision@3 on {sample_size} sampled test users …")
p3_hybrid   = precision_at_k(sample_test_df, recommend, k=3, threshold=7)
p3_baseline = precision_at_k(sample_test_df, baseline_recommend, k=3, threshold=7)

# ---------- 6) Print results ----------
print(f"\n{'Metric':<20} {'Hybrid':>10} {'Baseline':>10}")
print(f"{'-'*42}")
print(f"{'RMSE':<20} {rmse_hybrid:>10.4f} {rmse_baseline:>10.4f}")
print(f"{'Precision@3':<20} {p3_hybrid:>10.4f} {p3_baseline:>10.4f}")

# ---------- 7) 5-fold cross-validation (SVD on centered ratings) ----------
print("\n── 5-Fold Cross-Validation (SVD on centered ratings) ─────────────────")
kf = SurpriseKFold(n_splits=5, random_state=42, shuffle=True)
cv_rmses = []
for fold_i, (cv_trainset, cv_testset) in enumerate(kf.split(surprise_data_centered), 1):
    cv_svd = SVD(
        n_factors=best_params['n_factors'],
        n_epochs=best_params['n_epochs'],
        lr_all=best_params['lr_all'],
        reg_all=best_params['reg_all'],
        random_state=42
    )
    cv_svd.fit(cv_trainset)
    predictions = cv_svd.test(cv_testset)
    fold_rmse = accuracy.rmse(predictions, verbose=False)
    cv_rmses.append(fold_rmse)
    print(f"  Fold {fold_i}: RMSE = {fold_rmse:.4f}")

print(f"\n  Mean CV RMSE: {np.mean(cv_rmses):.4f} ± {np.std(cv_rmses):.4f}")

Computing RMSE on the full test set …
Computing Precision@3 on 500 sampled test users …

Metric                   Hybrid   Baseline
------------------------------------------
RMSE                     1.7146     1.7146
Precision@3              0.0623     0.0561

── 5-Fold Cross-Validation (SVD on centered ratings) ─────────────────
  Fold 1: RMSE = 1.4273
  Fold 2: RMSE = 1.4438
  Fold 3: RMSE = 1.4201
  Fold 4: RMSE = 1.4207
  Fold 5: RMSE = 1.4101

  Mean CV RMSE: 1.4244 ± 0.0112


## 10. Example Recommendations

Show the top-3 recommendations for several users, including their predicted hybrid score and the genres of each recommended movie.

In [ ]:
# ── Example recommendations for several users ────────────────────────────────
# Pick 5 users with varying activity levels
active_users = train_df['user_id'].value_counts()
example_users = list(active_users.index[:3])          # 3 most active users

# Also include a less-active user (fewest ratings among valid users)
least_active = active_users.tail(2).index.tolist()
example_users.extend(least_active)

for uid in example_users:
    n_rated = user_train_counts.get(uid, 0)
    tag = "  ← cold-start (popularity)" if uid in cold_users else ""
    print(f"\n{'='*72}")
    print(f"  User {uid}   ({n_rated} training ratings){tag}")
    print(f"{'='*72}")
    recs = recommend(uid, n=3)
    for _, row in recs.iterrows():
        print(f"  #{int(row['rank'])}  {row['title']:<50s}  score={row['score']:.3f}")
        print(f"      Genres: {row['genres']}")


  User 3   (5 training ratings)  ← cold-start (popularity)
  #1  Interstellar                                        score=57.068
      Genres: Adventure, Drama, Sci-Fi
  #2  Django Unchained                                    score=54.784
      Genres: Drama, Western
  #3  Gravity                                             score=53.142
      Genres: Drama, Sci-Fi, Thriller

  User 4   (5 training ratings)  ← cold-start (popularity)
  #1  1917                                                score=58.907
      Genres: Drama, War
  #2  Interstellar                                        score=57.068
      Genres: Adventure, Drama, Sci-Fi
  #3  Django Unchained                                    score=54.784
      Genres: Drama, Western

  User 5   (5 training ratings)  ← cold-start (popularity)
  #1  1917                                                score=58.907
      Genres: Drama, War
  #2  Interstellar                                        score=57.068
      Genres: Adventure, Dra

: 

## Bronnen

[1] K. Falk, Practical Recommender Systems. Manning Publications, 2019.